# పాఠం 11 - మోడల్ కాంటెక్స్ట్ ప్రోటొకాల్ (MCP)

**మోడల్ కాంటెక్స్ట్ ప్రోటొకాల్ (MCP)** ఒక ఓపెన్ స్టాండర్డ్, ఇది ఏజెంట్లకు రన్‌టైమ్‌లో గతిశీలంగా టూల్‌లు, వనరులు మరియు డేటా మూలాలను కనుగొని ఉపయోగించుకోవడానికి అనుమతిస్తుంది.

ఒక ఏజెంట్‌లో టూల్‌లను హార్డ్‌కోడ్ చేయడానికి బదులుగా, MCP ఏజెంట్లు అవసరం వచ్చినప్పుడు సామర్థ్యాలను అందించే బాహ్య సర్వర్లకు కనెక్ట్ అవ్వగలవు.

ఈ పాఠంలో మీరు నేర్చుకుంటారు:
- MCP అంటే ఏమిటి మరియు అది ఏజెంట్ వ్యవస్థల కోసం ఎందుకు ప్రాముఖ్యం
- MCP క్లయింట్-సర్వర్ ఆర్కిటెక్చర్ ఎలా పనిచేస్తుంది
- MCP-శైలి సాధన కనుగొనికను ఉపయోగించే ఏజెంట్లను ఎలా నిర్మించాలో


## సెటప్

**ముందస్తు అవసరాలు:**
- డిప్లాయ్ చేసిన మోడల్ ఉన్న Azure AI Foundry ప్రాజెక్ట్
- ప్రామాణీకరణ కోసం `az login` నడపండి


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) అంటే ఏమిటి?

MCP బాహ్య టూల్స్ మరియు డేటా మూలాలను కనుగొని వాటితో పరస్పర సంబంధం ఏర్పరచుకోవడానికి ఒక ప్రామాణిక విదానాన్ని నిర్వచిస్తుంది:

- **MCP Server**: ఒక ప్రామాణిక ప్రోటోకాల్ ద్వారా టూల్స్, వనరులు మరియు ప్రాంప్ట్‌లను బహిర్గతం చేస్తుంది
- **MCP Client**: సర్వర్లతో కనెక్ట్ అయి అందుబాటులో ఉన్న సామర్థ్యాలను కనుగొనే ఏజెంట్ రన్‌టైమ్
- **Dynamic Discovery**: ఏజెంట్లకు హార్డ్‌కోడ్ చేసిన టూల్స్ అవసరం ఉండదు — అవి రన్‌టైమ్‌లో ఏమి అందుబాటులో ఉందో కనుగొంటాయి

ఇది ఏజెంట్ కోడ్‌ను సవరించవలసి లేకుండానే కొత్త సామర్థ్యాలను జోడించగలిగే విస్తరించదగిన ఏజెంట్ వ్యవస్థలను నిర్మించడంలో శక్తివంతంగా ఉంటుంది.


## MCP ఎలా పనిచేస్తుంది

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ఏజెంట్ (MCP client) ఒక MCP serverకి కనెక్ట్ అవుతుంది
2. సర్వర్ అందుబాటులో ఉన్న ఉపకరణాల జాబితా మరియు వాటి స్కీమాలతో స్పందిస్తుంది
3. ఆ తర్వాత ఏజెంట్ తన తర్క ప్రక్రియలో కనుగొన్న ఏ ఉపకరణాన్ని అయినా పిలవవచ్చు
4. ఫలితాలు అదే ప్రోటోకాల్ ద్వారా తిరిగి వస్తాయి


## MCP టూల్ కనుగొనడాన్ని అనుకరించడం

నిజమైన MCP సర్వర్ ఒక నడుస్తున్న సర్వర్ ప్రక్రియను అవసరం పడినందున, MCP-కు కనెక్ట్ అయిన ఒక వసతి సేవ ఏమి అందించేది అనుకరించేలా `@tool` ఫంక్షన్లను ఉపయోగించి ఆ నమూనాను ప్రదర్శిస్తాము.

ఉత్పత్తి వాతావరణంలో, ఈ టూల్‌లు స్థానికంగా నిర్వచించబడే బదులు MCP సర్వర్ నుండి డైనమిక్‌గా కనుగొనబడతాయి.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-శైలి సాధనాలతో ఏజెంట్ నిర్మించడం


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ఉత్పత్తి వాతావరణంలో MCP

ఉత్పత్తి వాతావరణంలో, MCP శక్తివంతమైన నమూనాలను అనుమతిస్తుంది:

- **Dynamic tool discovery**: ఏజెంట్లు MCP సర్వర్లకు కనెక్ట్ అవుతూ రన్‌టైమ్‌లో టూల్‌లను కనుగొంటాయి
- **Decoupled architecture**: టూల్ ప్రొవైడర్లు ఏజెంట్ నుంచి స్వతంత్రంగా నవీకరించుకోవచ్చు
- **Cross-organization sharing**: టీములు MCP సర్వర్ల ద్వారా సామర్థ్యాలను అందుబాటులో ఉంచవచ్చు, వాటిని ఏ ఏజెంట్ అయినా ఉపయోగించవచ్చు
- **Microsoft Agent Framework support**: MAFకి బిల్ట్-ఇన్ MCP క్లయింట్ మద్దతు ఉంది `mcp` ఇంటిగ్రేషన్ ద్వారా

To use a real MCP server with MAF, you would connect via `hosted_mcp_tool()` or the MCP client integration.

**మరింత తెలుసుకోండి:**
- [MCP స్పెసిఫికేషన్](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP మద్దతు](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## సారాంశం

ఈ పాఠంలో, మీరు నేర్చుకున్నవి:
- **MCP** అనేది ఏజెంట్లు మరియు టూల్ ప్రొవైడర్ల మధ్య డైనమిక్ టూల్ గుర్తింపుకు ఉపయోగించే ఓపెన్ స్టాండర్డ్
- **క్లయింట్-సర్వర్ ఆర్కిటెక్చర్** ఏజెంట్లు రన్‌టైమ్ సమయంలో సామర్థ్యాలను కనుగొనగలిగేలా చేస్తుంది
- MCP ద్వారా కోడ్ మార్పులు లేకుండా టూల్స్ జోడించగల **విస్తరింపదగిన, వేరుచేసిన ఏజెంట్ వ్యవస్థలు** సాధ్యమవుతాయి
- Microsoft Agent Framework ఉత్పత్తి వినియోగానికి **నిర్మిత MCP మద్దతును** అందిస్తుంది


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
నిరాకరణ:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ద్వారా అనువదించబడింది. మేము ఖచ్చితత్వాన్ని శ్రద్ధగా పాటించేందుకు ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో పొరపాట్లు లేదా లోపాలు ఉండే అవకాశం ఉందని దయచేసి గమనించండి. మూల పత్రాన్ని దాని స్వదేశీ భాషలోనే అధికారిక మూలంగా పరిగణించాలి. అత్యవసర లేదా ముఖ్యం సమాచారం కోసం వృత్తిపరమైన మానవ అనువాదాన్ని చేయించుకోవాలని సిఫార్సు చేయబడుతుంది. ఈ అనువాదాన్ని ఉపయోగించడం వలన కలిగే ఏవైనా అవగాహనా లోపాలు లేదా తప్పుగా అర్థం చేసుకోవడాల కోసం మేము బాధ్యులు కాదని తెలియజేస్తున్నాము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
